# Homework: dlt

## Question 1. Instrument the agent with Logfire

Answer **15**

## Question 2. Load traces into DuckDB with dlt

In [45]:
DB_PATH = 'agent_traces_pipeline.duckdb'

In [46]:
import duckdb

QUERY = """
SELECT COUNT(*) FROM information_schema.tables 
WHERE table_schema = 'agent_traces';
"""

con = duckdb.connect(DB_PATH)
result = con.execute(QUERY).fetchdf()
con.close()
result

,count_star()
0,10


## Question 3. Query traces with an agent
Using a coding agent (you can also write the code by hand) find the input token usage for the agent run from Q1.



In [47]:
import duckdb

QUESTION = 'How do I run Ollama locally?'

QUERY = """
WITH llm_calls AS (
    SELECT
        created_at,
        COALESCE(
            attributes__result__usage__prompt_tokens,
            attributes__input_data__usage__prompt_tokens,
            attributes__result__prompt_tokens,
            attributes__input_data__prompt_tokens
        ) AS input_tokens
    FROM agent_traces.query_rows
    WHERE attributes__code_filepath LIKE '%main.py%'
      AND message ILIKE '%ChatCompletion%'
      AND created_at >= (CURRENT_TIMESTAMP - INTERVAL '5 minute')
),
ordered AS (
    SELECT *, LAG(created_at) OVER (ORDER BY created_at) AS prev_created_at
    FROM llm_calls
),
sessionized AS (
    SELECT
        *,
        SUM(
            CASE
                WHEN prev_created_at IS NULL THEN 1
                WHEN date_diff('second', prev_created_at, created_at) > 15 THEN 1
                ELSE 0
            END
        ) OVER (ORDER BY created_at) AS run_id
    FROM ordered
),
run_totals AS (
    SELECT
        run_id,
        MIN(created_at) AS run_start,
        MAX(created_at) AS run_end,
        COUNT(*) AS llm_calls,
        SUM(input_tokens) AS total_input_tokens
    FROM sessionized
    GROUP BY run_id
),
anchors AS (
    SELECT DISTINCT q.created_at
    FROM agent_traces.query_rows q
    JOIN agent_traces.query_rows__attributes__input_data i
      ON i._dlt_parent_id = q._dlt_id
    WHERE i.function__name = 'search'
      AND i.function__arguments ILIKE '%ollama%'
      AND q.created_at >= (CURRENT_TIMESTAMP - INTERVAL '5 minute')
),
target_runs AS (
    SELECT DISTINCT r.*
    FROM run_totals r
    JOIN anchors a
      ON a.created_at BETWEEN r.run_start - INTERVAL '2 second' AND r.run_end + INTERVAL '2 second'
    ORDER BY r.run_end DESC
)
SELECT * FROM target_runs;
"""

con = duckdb.connect(DB_PATH)
runs_df = con.execute(QUERY).fetchdf()
con.close()

if runs_df.empty:
    print('No matching Ollama run found in the last 5 minutes.')
else:
    latest_run = runs_df.iloc[0]
    total_input_tokens = int(latest_run['total_input_tokens'])

    lower = (total_input_tokens // 1000) * 1000
    upper = lower + 1000
    token_range = f"{lower}-{upper}"

    print(f"Question topic: {QUESTION}")
    print(f"Input tokens total: {total_input_tokens}")
    print(f"Range: {token_range}")

runs_df

Question topic: How do I run Ollama locally?
Input tokens total: 4099
Range: 4000-5000


,run_id,run_start,run_end,llm_calls,total_input_tokens
0,1.0,2026-07-18 16:11:55.718231+00:00,2026-07-18 16:11:58.448322+00:00,3,4099.0
